In [ ]:
# 1 — cfg
import os, sys, json, shutil
from pathlib import Path

DRIVE_FOLDER = "ProtoFormer_out"
REPO = "/content/ProtoFormer"
GIT_URL = "https://github.com/pratamabintang/PF.git"
BRANCH = "main"
DATA = Path("/content/data")
SAM2_CKPT = "/content/sam2_hiera_l.pth"
SAM2_DRIVE_SRC = "/content/drive/MyDrive/project/checkpoints/sam2_hiera_large.pt"

# hyperparameter — ubah di sini saja
HP = dict(
    benchmark="pascal", fold=0,
    bsz=16, lr=1e-3, niter=3000, nworker=2, reduce_dim=64,
    amp="auto", patience=20, val_interval=3,
)

print(HP)


In [ ]:
# 2 — drive + repo + deps
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
OUT = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
OUT.mkdir(parents=True, exist_ok=True)
try:
    import git
except ImportError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "GitPython"], check=True)
    import git
print("OUT:", OUT)


In [ ]:
# 3 — repo clone/pull
if not Path(REPO).exists():
    git.Repo.clone_from(GIT_URL, REPO, branch=BRANCH)
else:
    repo = git.Repo(REPO)
    repo.git.fetch("origin")
    repo.git.checkout(BRANCH)
    repo.git.reset("--hard", f"origin/{BRANCH}")
    repo.git.clean("-fd")
os.chdir(REPO)
assert Path(REPO, "train.py").exists()
print("repo OK", REPO, "sam2 ada:", Path(REPO, "model/sam2").exists())

# logs/ persisten: simpan checkpoint & log ke Drive, bukan ke /content yang hilang saat runtime recycle
DRIVE_LOGS = OUT / "logs"
DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
LOCAL_LOGS = Path(REPO) / "logs"
if LOCAL_LOGS.is_symlink() or LOCAL_LOGS.is_file():
    LOCAL_LOGS.unlink()
elif LOCAL_LOGS.exists():
    # pindahkan artefak lama ke Drive sebelum dijadikan symlink
    for _p in list(LOCAL_LOGS.rglob("*")):
        if _p.is_file():
            _dst = DRIVE_LOGS / _p.relative_to(LOCAL_LOGS)
            _dst.parent.mkdir(parents=True, exist_ok=True)
            if not _dst.exists():
                shutil.copy2(_p, _dst)
    shutil.rmtree(LOCAL_LOGS)
LOCAL_LOGS.symlink_to(DRIVE_LOGS, target_is_directory=True)
print("logs -> Drive:", LOCAL_LOGS, "->", DRIVE_LOGS)


In [ ]:
# 4 — deps
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import torch, torchvision
print(torch.__version__, torchvision.__version__)


In [ ]:
# 5 — VOC2012 dari HuggingFace (tanpa subprocess)
from datasets import load_dataset
voc = DATA / "VOC2012"
for d in (voc / "JPEGImages", voc / "SegmentationClassAug", voc / "SegmentationClass"):
    d.mkdir(parents=True, exist_ok=True)
jpg_dir, seg_dir, img_dir = (voc / "JPEGImages", voc / "SegmentationClassAug", voc / "SegmentationClass")
if len(list(jpg_dir.glob("*.jpg"))) < 1000 or len(list(seg_dir.glob("*.png"))) < 1000:
    ds = load_dataset("shijli/voc2012", "segmentation_aug")
    seen = set()
    for split_name in ("train", "validation"):
        sp = ds[split_name]
        for i in range(len(sp)):
            row = sp[i]
            img_id = row["id"]
            if img_id in seen:
                continue
            seen.add(img_id)
            row["image"].convert("RGB").save(str(jpg_dir / f"{img_id}.jpg"), "JPEG", quality=95)
            row["mask"].save(str(seg_dir / f"{img_id}.png"), "PNG")
    ds2 = load_dataset("shijli/voc2012", "segmentation")
    for split_name in ("train", "validation"):
        sp = ds2[split_name]
        for i in range(len(sp)):
            row = sp[i]
            row["mask"].save(str(img_dir / f"{row['id']}.png"), "PNG")
    del ds, ds2
n_jpg = len(list(jpg_dir.glob("*.jpg")))
n_aug = len(list(seg_dir.glob("*.png")))
n_off = len(list(img_dir.glob("*.png")))
assert n_jpg >= 11831, f"JPEGImages kurang: {n_jpg}"
assert n_aug >= 11831, f"SegmentationClassAug kurang: {n_aug}"
assert n_off >= 2813, f"SegmentationClass kurang: {n_off}"
print("VOC OK", n_jpg, n_aug, n_off)


In [ ]:
# 6 — SAM2 checkpoint dari Drive
if Path(SAM2_DRIVE_SRC).exists():
    shutil.copy(SAM2_DRIVE_SRC, SAM2_CKPT)
assert Path(SAM2_CKPT).exists() and Path(SAM2_CKPT).stat().st_size > 100 * 1024 * 1024, \
    f"SAM2 rusak: {Path(SAM2_CKPT).stat().st_size if Path(SAM2_CKPT).exists() else 'missing'} byte"
print("SAM2 OK", Path(SAM2_CKPT).stat().st_size // 1024 // 1024, "MB")


In [ ]:
# 7 — train 1-shot (CLI, foreground — kernel block sampai selesai)
# PYTHONPATH=model: sam2 vendored di model/sam2, bukan pip package
!cd /content/ProtoFormer && PYTHONPATH=model python train.py --datapath /content/data --benchmark pascal --fold 0 --bsz 16 --lr 0.001 --niter 3000 --nworker 2 --reduce_dim 64 --img_size 512 --amp auto --sam2_checkpoint /content/sam2_hiera_l.pth --patience 20 --val_interval 3 --shot 1 --resume --logpath pascal_fold0_1shot


In [ ]:
# 8 — train 5-shot (CLI, foreground — kernel block sampai selesai)
# PYTHONPATH=model: sam2 vendored di model/sam2, bukan pip package
!cd /content/ProtoFormer && PYTHONPATH=model python train.py --datapath /content/data --benchmark pascal --fold 0 --bsz 16 --lr 0.001 --niter 3000 --nworker 2 --reduce_dim 64 --img_size 512 --amp auto --sam2_checkpoint /content/sam2_hiera_l.pth --patience 20 --val_interval 3 --shot 5 --resume --logpath pascal_fold0_5shot


In [ ]:
# 9 — save: salin weight + histori ke Drive
import zipfile

# Logger menyimpan ke logs/<logpath>.log/best_model.pt  (dan last_model.pt untuk resume)
# rglob dipakai supaya tidak bergantung nama folder maupun nama file lama (best_*.pth sudah tidak dipakai)
REPO_P = Path(REPO)
# hanya ambil dari logs/ -> tidak ikut kernel-metadata.json dsb di luar training
# pola .pt/.pth mencakup nama lama (best_*.pth) dan baru (best_model.pt / last_model.pt),
# supaya weight hasil kode sebelum commit ini tetap ikut tersimpan
artifacts = []
logs_root = REPO_P / "logs"
for pat in ("**/*.pt", "**/*.pth", "**/log.txt", "**/events.out.*"):
    artifacts += [p for p in logs_root.rglob(pat) if p.is_file()]

# weight lama yang mungkin ditaruh di root repo (mis. best_model.pt hasil run sebelumnya)
for pat in ("best_*.pth", "best_model.pt"):
    artifacts += [p for p in REPO_P.glob(pat) if p.is_file()]

# dedupe, buang artefak tbd/runs supaya zip tidak membengkak
seen, keep = set(), []
for p in artifacts:
    rp = p.resolve()
    if rp in seen or "tbd" in p.parts:
        continue
    seen.add(rp)
    keep.append(p)

print("ditemukan:", len(keep))
for p in keep:
    print("  ", p.relative_to(REPO_P), f"{p.stat().st_size / 1024 / 1024:.1f} MB")

# diagnostik kalau kosong: tampilkan isi logs/ apa adanya
if not keep:
    print("\n== isi logs/ (diagnostik) ==")
    logs = REPO_P / "logs"
    if not logs.exists():
        print("logs/ TIDAK ADA -> training tidak menulis log, atau runtime sudah di-recycle (logs/ tidak persisten di Colab)")
        print("cek juga: apakah training benar-benar selesai >= 1 epoch val (val_interval=%d)?" % HP["val_interval"])
    else:
        for p in sorted(logs.rglob("*"))[:200]:
            print("  ", p.relative_to(REPO_P), "(dir)" if p.is_dir() else f"{p.stat().st_size / 1024:.0f} KB")

assert keep, "tidak ada artefak tersimpan — lihat diagnostik di atas"

files = []
for p in keep:
    dst = OUT / p.name
    if dst.exists():
        dst = OUT / f"{p.parent.name}__{p.name}"  # hindari tabrakan antar-run
    shutil.copy(p, dst)
    files.append(dst)

with zipfile.ZipFile(OUT / "protoformer_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f.name)

print("\nsaved:", OUT / "protoformer_out.zip")
